# 🛡️ 도구 엔지니어링 & MCP (Tools & MCP)

본 노트북에서는 **Progressive Skill Disclosure(점진적 도구 노출)** 패턴을 분석하고, **MCP(Model Context Protocol)** 클라이언트-서버 stdio 연결을 통해 도구를 프로세스 수준으로 **정적 바인딩(Static Binding)**하는 방법과 시스템 프롬프트 컨텍스트로 활용하는 **동적 바인딩(Dynamic Context Binding)** 방식을 비교 실습합니다.

In [ ]:
# 1. 환경 변수 로드 및 초기화
import sys
import os
import re
from dotenv import load_dotenv

# ⚠️ 주피터 실행 디렉토리(notebooks/)와 프로젝트 루트(agent-harness-lab/) 경로 싱크 정합
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
    print(f"🔄 작업 디렉토리를 프로젝트 루트('{os.getcwd()}')로 전환 완료.\n")

# LangSmith API Key Forbidden 경고 차단
os.environ["LANGCHAIN_TRACING_V2"] = "false"

# 프로젝트 루트 경로 기준 src 추가 및 .env 수동 로드
sys.path.append(os.path.abspath("src"))
load_dotenv(override=True)

from utils.llm import get_llm
# AAWS 연동 텍스트 정규화 유틸리티 임포트
from utils.message_utils import normalize_content
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage


--- 
## 📌 Part 1. Skills & Progressive Disclosure (점진적 도구 노출)

### 1. Progressive Disclosure와 기본 도구(Base Tools)의 중요성
현업 에이전트는 50개 이상의 많은 도구를 한 번에 시스템 프롬프트에 주입하면 **도구 오호출(Tool Overload Penalty) 및 지연 시간(Latency)의 급증**을 겪습니다. 이를 해결하기 위해 에이전트에게 처음부터 모든 도구를 주는 것이 아니라, 필요에 따라 런타임에 동적으로 기능을 확장해 나가는 **Progressive Disclosure**를 채택합니다.

에이전트가 새로운 도구(Skill)를 런타임에 호출하고 탑재하기 위해서는 `bash_tool`, `write_file` 등 **운영체제와 연동할 수 있는 기본 도구(Base Tools)**가 필수적으로 선탑재되어 있어야 합니다.

#### 💡 기본 도구가 필수적인 이유
- **런타임 코드 작성 및 실행**: 에이전트가 직접 파이썬 스크립트 형식의 스킬을 `write_file`로 디스크에 쓰고, `bash`를 통해 서브프로세스로 실행하여 기능을 확장합니다.
- **외부 환경 격리**: 샌드박스 내부의 리소스를 연동하여 무한에 가까운 제어 능력을 확보하기 위함입니다.

### 📊 Skills 가동 흐름도

<div style="display: flex; flex-direction: column; align-items: center; justify-content: center; gap: 15px; font-family: 'Pretendard', sans-serif; background-color: #1e293b; padding: 25px; border-radius: 12px; border: 1px solid #334155; margin: 15px 0;">
  
  <!-- Prompt/Context (Blue) -->
  <div style="padding: 12px 18px; background-color: #eff6ff; color: #1e40af; border: 2px solid #3b82f6; border-radius: 8px; font-weight: bold; font-size: 0.85rem; box-shadow: 0 4px 6px rgba(0,0,0,0.15); text-align: center; min-width: 300px;">
    💬 User Request: "PDF 파일의 텍스트를 추출해 줘"
  </div>
  <div style="color: #94a3b8; font-size: 1.2rem;">⬇</div>
  
  <!-- Agent (Amber) -->
  <div style="padding: 12px 18px; background-color: #fff7ed; color: #9a3412; border: 2px solid #f97316; border-radius: 8px; font-weight: bold; font-size: 0.85rem; box-shadow: 0 4px 6px rgba(0,0,0,0.15); text-align: center; min-width: 300px;">
    🧠 Agent: "기본 도구(bash)를 사용하여 skills/pdf_extractor.py 호출"
  </div>
  <div style="color: #94a3b8; font-size: 1.2rem;">⬇</div>
  
  <!-- Tool/Result (Green) -->
  <div style="padding: 12px 18px; background-color: #f0fdf4; color: #166534; border: 2px solid #22c55e; border-radius: 8px; font-weight: bold; font-size: 0.85rem; box-shadow: 0 4px 6px rgba(0,0,0,0.15); text-align: center; min-width: 300px;">
    🛠️ Bash Output: 추출된 구조화된 JSON 데이터 획득
  </div>
  
</div>

In [ ]:
# Part 1. 기본 도구(Bash/FileRead)를 장착한 ReAct 에이전트의 스킬 자율 발견 및 격발
import os
from utils.llm import get_llm
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from harness.tools.claude_tools import file_read, bash_command
from langchain_core.messages import HumanMessage
from utils.test_log import stream_and_debug_agent

# 1. LLM 팩토리 로드 및 Claude Code 8대 기본 도구 세트 바인딩
llm = get_llm(model_name="gpt-4o", temperature=0.0)
tools = [file_read, bash_command]

# 2. 에이전트의 Progressive Disclosure 가이드 시스템 지침 정의
system_instruction = (
    "You are a progressive skill-disclosure agent.\n"
    "You only have 'file_read' and 'bash_command' base tools.\n"
    "Your objective is to find and execute a custom sub-script under 'skills/pdf_processing/' that handles the user's PDF task.\n"
    "Follow these steps:\n"
    "1. Read 'skills/pdf_processing/Skill.md' using 'file_read' to learn about available scripts and arguments.\n"
    "2. Choose the correct script (pdf_extractor, pdf2text, or pdf2image) and execute it via 'bash_command'.\n"
    "Output the final results exactly as returned by the script.\n"
    "Ensure you use relative paths correctly relative to the project root directory."
)

# 3. LangChain 정식 create_agent API로 ReAct 에이전트 구축
checkpointer = MemorySaver()
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_instruction,
    checkpointer=checkpointer
)

# 4. 에이전트 구동 및 [stream_and_debug_agent] 디버거로 자율 탐색 핑퐁 관찰
user_request = (
    "notebooks/pdf/sample.pdf 파일에서 텍스트와 메타데이터를 추출해줘. "
    "먼저 관련 스킬의 Skill.md 파일을 읽고 사용법을 파악한 뒤 bash_command로 스크립트를 기동해야 해."
)

config = {"configurable": {"thread_id": "skill_disclosure_session"}}
inputs = {"messages": [HumanMessage(content=user_request)]}

# 통합 디버거 헬퍼 기동 (실시간 스트리밍 & 전체 궤적 정밀 덤프 일괄 출력)
stream_and_debug_agent(agent, inputs, config, agent_name="Progressive Skill 에이전트")


--- 
## 📌 Part 2. MCP (Model Context Protocol) 연동 및 바인딩 패턴 비교

Model Context Protocol(MCP)은 에이전트와 외부 데이터 소스/도구 서버 간의 표준 양방향 통신 규격입니다. 본 실습에서는 파이썬으로 구현된 SQLite MCP 서버와 통신하는 2가지 방식을 비교합니다.

### 1. Static Process Binding vs Dynamic Context Binding

| 비교 항목 | Static Process Binding (정적 바인딩) | Dynamic Context Binding (동적 바인딩) |
|---|---|---|
| **개념** | MCP 서버의 도구를 로드하여 에이전트 **프로세스 자체**에 직접 결합 | 도구를 프로세스가 아닌 에이전트 **프롬프트 컨텍스트**로 다룸 |
| **구현 방식** | `langchain-mcp-adapters`를 통해 LangChain `StructuredTool`로 변환 및 주입 | MCP Client가 `list_tools`로 스키마 조회(Discovery) -> 프롬프트 주입 -> 필요 시 API 호출(Invoke) |
| **효율성** | 도구 수가 적을 때 빠르고 편리하지만, 도구 과부하(Overload) 지연 발생 | 도구가 무수히 많아도 API 명세만 프롬프트에 동적 요약해 올리므로 **매우 가볍고 효율적** |

### 📊 MCP Client-Server Stdio 아키텍처

<div style="display: flex; align-items: center; justify-content: center; gap: 15px; font-family: 'Pretendard', sans-serif; background-color: #1e293b; padding: 25px; border-radius: 12px; border: 1px solid #334155; margin: 15px 0;">
  
  <!-- Agent (Amber) -->
  <div style="padding: 12px 18px; background-color: #fff7ed; color: #9a3412; border: 2px solid #f97316; border-radius: 8px; font-weight: bold; font-size: 0.85rem; box-shadow: 0 4px 6px rgba(0,0,0,0.15); text-align: center; min-width: 150px;">
    🧠 LLM Agent
  </div>
  <div style="color: #94a3b8; font-size: 1.5rem; font-weight: bold;">⇄</div>
  
  <!-- Client Bridge (Blue) -->
  <div style="padding: 12px 18px; background-color: #eff6ff; color: #1e40af; border: 2px solid #3b82f6; border-radius: 8px; font-weight: bold; font-size: 0.85rem; box-shadow: 0 4px 6px rgba(0,0,0,0.15); text-align: center; min-width: 150px;">
    🔌 MCP Client
  </div>
  <div style="color: #94a3b8; font-size: 1.2rem;"> (Stdio 파이프) ⇄ </div>
  
  <!-- Server (Green) -->
  <div style="padding: 12px 18px; background-color: #f0fdf4; color: #166534; border: 2px solid #22c55e; border-radius: 8px; font-weight: bold; font-size: 0.85rem; box-shadow: 0 4px 6px rgba(0,0,0,0.15); text-align: center; min-width: 150px;">
    🗄️ SQLite MCP Server
  </div>
  
</div>

Static Process Binding (정적 바인딩) 방식으로 MCP 서버에 연결해보겠습니다.

In [ ]:
# Part 2-1. 전용 MCP 도구(list/execute)를 에이전트 프로세스에 직접 연동하여 Wikipedia 실시간 검색
import os
import asyncio
import nest_asyncio
from utils.llm import get_llm
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from utils.test_log import stream_and_debug_agent
from fastmcp import Client
from fastmcp.client.transports import StdioTransport

# ⚠️ 주피터 커널 비동기 락 방지용 가드레일 적용
nest_asyncio.apply()

# ── notebooks/prompts/MCP.md 경로 ──
_PROJECT_ROOT = os.getcwd()
MCP_CONTEXT_PATH = os.path.join(_PROJECT_ROOT, "notebooks", "prompts", "MCP.md")

# 시동 시점에 시스템 프롬프트(컨텍스트)로 주입할 notebooks/prompts/MCP.md 파일 내용 미리 읽기
mcp_context_str = ""
if os.path.exists(MCP_CONTEXT_PATH):
    with open(MCP_CONTEXT_PATH, "r", encoding="utf-8") as f:
        mcp_context_str = f.read()
else:
    mcp_context_str = "No active MCP servers registered in catalog."


# =============================================================================
# 🛠️ 1. 하이브리드 통신(SSE/Stdio)을 지원하는 전용 MCP 도구 2종 정의
# =============================================================================

def get_mcp_client(target: str) -> Client:
    if target.startswith("http://") or target.startswith("https://"):
        return Client(target)
    else:
        import shlex
        parts = shlex.split(target)
        transport = StdioTransport(command=parts[0], args=parts[1:])
        return Client(transport)


@tool
def list_mcp_tools(url: str) -> str:
    """지정된 MCP 서버 URL 또는 구동 커맨드(url)가 실시간으로 제공하는 모든 도구 목록과 파라미터 스키마를 가져옵니다.

    Args:
        url: 조회하고자 하는 MCP 서버의 전체 HTTP/SSE 엔드포인트 주소 또는 Stdio 구동 명령어 (예: 'npx -y wikipedia-mcp')
    """
    async def _run():
        client = get_mcp_client(url)
        async with client:
            tools_response = await client.list_tools()
            tools_list = []
            for t in tools_response:
                tools_list.append({
                    "name": t.name,
                    "description": t.description,
                    "input_schema": t.inputSchema
                })
            return str(tools_list)
            
    try:
        return asyncio.run(_run())
    except Exception as e:
        return f"MCP 서버 '{url}'로부터 도구 목록을 가져오는 데 실패했습니다: {str(e)}"


@tool
def execute_mcp_tool(url: str, tool_name: str, arguments: dict) -> str:
    """지정된 MCP 서버 URL 또는 구동 커맨드(url)의 특정 도구(tool_name)를 입력한 인자(arguments)로 실행하고 결과를 반환합니다.

    Args:
        url: 실행할 도구가 위치한 MCP 서버의 전체 HTTP/SSE 엔드포인트 주소 또는 Stdio 구동 명령어 (예: 'npx -y wikipedia-mcp')
        tool_name: 실행하고자 하는 도구 이름
        arguments: 도구 실행에 필요한 파라미터 딕셔너리
    """
    async def _run():
        client = get_mcp_client(url)
        async with client:
            result = await client.call_tool(tool_name, arguments)
            
            content_str = str(result.content)
            try:
                if hasattr(result, "content") and isinstance(result.content, list):
                    content_str = "\n".join([item.text for item in result.content if hasattr(item, "text")])
            except Exception:
                pass
            return content_str
            
    try:
        return asyncio.run(_run())
    except Exception as e:
        return f"MCP 서버 '{url}'의 도구 '{tool_name}' 실행 중 에러 발생: {str(e)}"


# =============================================================================
# 🤖 2. Direct Binding ReAct 에이전트 구축
# =============================================================================
llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)
tools = [list_mcp_tools, execute_mcp_tool]

system_instruction = (
    "You are a helpful assistant equipped with Model Context Protocol (MCP) interface tools.\n"
    "Your objective is to solve user requests by connecting to the correct MCP servers.\n\n"
    "Here is the context of active MCP servers you can use:\n"
    f"====== ACTIVE MCP SERVERS CATALOG ======\n{mcp_context_str}\n========================================\n\n"
    "Follow these steps sequentially to run a tool:\n"
    "1. Look at the CATALOG to identify the correct MCP server target URL or command for the user request.\n"
    "2. Call 'list_mcp_tools' using that target to learn about its available tools and schemas.\n"
    "3. Call 'execute_mcp_tool' with the target, tool_name, and required arguments to execute the tool.\n"
    "Finally, summarize and return the retrieved information clearly to the user."
)

checkpointer = MemorySaver()
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_instruction,
    checkpointer=checkpointer
)

# =============================================================================
# 🚀 3. Wikipedia 검색 기동
# =============================================================================
user_request = (
    "wikipedia-mcp 서버에서 "
    "실시간으로 제공하는 도구를 조회한 뒤, 딥러닝 키워드로 위키피디아를 검색하고 요약해서 알려줘."
)

config = {"configurable": {"thread_id": "direct_mcp_session"}}
inputs = {"messages": [HumanMessage(content=user_request)]}

# 통합 디버거 헬퍼 기동
stream_and_debug_agent(agent, inputs, config, agent_name="Direct MCP Binding 에이전트")


Dynamic Context Binding (동적 바인딩)은 도구를 컨텍스트로서 인식합니다. 에이전트 입장에서 난이도는 있지만 더 확장성있는 방식입니다.

In [ ]:
# Part 2-2. 기본 도구(FileRead/Bash)와 Skills 경로 주입만을 활용한 Wikipedia 자율 검색
import os
from utils.llm import get_llm
from langchain.agents import create_agent
from langgraph.checkpoint.memory import MemorySaver
from harness.tools.claude_tools import file_read, bash_command
from langchain_core.messages import HumanMessage
from utils.test_log import stream_and_debug_agent

# 1. LLM 팩토리 로드 및 오직 2대 기본 도구 세트만 바인딩 (MCP 전용 툴 주입 불필요!)
llm = get_llm(model_name="gemini-3.5-flash", temperature=0.0)
tools = [file_read, bash_command]

# 2. ── notebooks/prompts/MCP.md 경로 ──
_PROJECT_ROOT = os.getcwd()
MCP_CONTEXT_PATH = os.path.join(_PROJECT_ROOT, "notebooks", "prompts", "MCP.md")

# 시동 시점에 시스템 프롬프트(컨텍스트)로 주입할 notebooks/prompts/MCP.md 파일 내용 미리 읽기
mcp_context_str = ""
if os.path.exists(MCP_CONTEXT_PATH):
    with open(MCP_CONTEXT_PATH, "r", encoding="utf-8") as f:
        mcp_context_str = f.read()
else:
    mcp_context_str = "No active MCP servers registered in catalog."

# 3. 에이전트의 Progressive Disclosure 가이드 및 Skills 우선순위 지침 정의
system_instruction = (
    "You are a progressive skill-disclosure agent.\n"
    "You only have 'file_read' and 'bash_command' base tools.\n"
    "Your objective is to solve user requests by discovering and running custom scripts.\n\n"
    "Here is the catalog of active MCP servers you can use:\n"
    f"====== ACTIVE MCP SERVERS CATALOG ======\n{mcp_context_str}\n========================================\n\n"
    "Guidelines for Skill Priority:\n"
    "- ALWAYS prioritize searching and utilizing custom scripts under the 'skills/' folder over running raw shell commands directly in 'bash_command'.\n"
    "- If you need to interact with any MCP servers listed in the CATALOG, you must inspect the guidelines at: 'skills/mcp/Skill.md' using 'file_read' first.\n"
    "- Follow the instructions in the Skill.md file to discover and execute the corresponding scripts to fulfill the request.\n"
    "Output the final results clearly to the user."
)

# 4. LangChain 정식 create_agent API로 ReAct 에이전트 구축
checkpointer = MemorySaver()
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_instruction,
    checkpointer=checkpointer
)

# 5. 에이전트 구동 및 [stream_and_debug_agent] 디버거로 Wikipedia 자율 탐색 핑퐁 관찰
user_request = (
    "wikipedia-mcp 서버에서 "
    "실시간으로 제공하는 도구를 조회한 뒤, Deep learning 키워드로 위키피디아를 검색하고 요약해서 알려줘."
)

config = {"configurable": {"thread_id": "progressive_mcp_session"}}
inputs = {"messages": [HumanMessage(content=user_request)]}

# 통합 디버거 헬퍼 기동 (실시간 2단계 Skills Discovery & Execution 궤적 관찰)
stream_and_debug_agent(agent, inputs, config, agent_name="Progressive MCP 에이전트")
